# DDS-Edge: huấn luyện, phân tích lỗi và lượng tử hoá trên Google Colab

Notebook này chạy trên bộ dữ liệu **đã làm sạch** `sf_320x240` (20.007 ảnh 320×240, đã bỏ ảnh gần trùng, chia theo tài xế 18/4/4).

**Thứ tự làm việc (chạy từng cell từ trên xuống):**

| Phần | Nội dung | Cell |
|---|---|---|
| A | Chuẩn bị: GPU, Drive, code, dữ liệu, thư mục lưu kết quả | 1 đến 5 |
| B | Chạy thử 1 epoch để chắc mọi thứ ổn | 6 |
| C | Train baseline và thử số epoch lớn hơn | 7 đến 8 |
| D | **Phân tích lỗi**: confusion matrix, điểm từng tài xế, xem ảnh bị nhầm | 9 đến 10 |
| E | **Cải thiện**: mất cân bằng lớp, augmentation mạnh hơn, độ phân giải ảnh vào | 11 |
| F | Kiểm tra `best.pt` đã có chưa (bắt buộc trước khi lượng tử hoá) | 12 |
| G | **Lượng tử hoá INT8** và đo độ trễ, FPS | 13 đến 14 |
| H | Bảng và biểu đồ tổng hợp | 15 |

**Cần chuẩn bị trước:** upload file `sf_320x240.zip` (hoặc `.rar`) lên *My Drive*; chọn **Runtime → Change runtime type → T4 GPU**.

**Kết quả được lưu thẳng vào Google Drive** (`MyDrive/dds-edge-runs/<tên run>/`), nên không mất khi Colab ngắt phiên. Sau khi ngắt, chạy lại cell 1 đến 5 rồi chạy tiếp.

**Cách dùng các cell có dấu `#` ở đầu dòng lệnh:** đó là các lệnh để sẵn. Muốn chạy lệnh nào thì bỏ `#` ở đúng dòng `# !python ...` đó (giữ nguyên `#` ở các dòng giải thích). Mỗi lần chỉ nên bỏ `#` một dòng để xem kết quả từng bước.

## A. Chuẩn bị môi trường

### Cell 1: kiểm tra GPU
Xác nhận Colab đang cấp GPU và in ra tên card (thường là `Tesla T4`). Nếu báo lỗi, vào **Runtime → Change runtime type**, chọn T4 GPU rồi chạy lại.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Bat GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.cuda.get_device_name(0))

### Cell 2: nối Google Drive
Gắn Drive vào Colab (sẽ hiện hộp thoại xin quyền, bạn bấm đồng ý). Drive dùng để đọc file dữ liệu `.zip` và lưu kết quả train.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Cell 3: tải code từ GitHub và cài thư viện
`git clone` lần đầu, các lần sau `git pull` để lấy code mới nhất. Cài thêm `timm` (cho EfficientNet-Lite0), `onnx` và `onnxruntime` (cho lượng tử hoá), `tabulate` (cho bảng tổng hợp). Dòng cuối in ra commit đang dùng, để bạn biết code đã được cập nhật chưa.

**Code mới (phân tích lỗi, sampler cân bằng, augmentation mạnh, lượng tử hoá nhiều kiểu, bảng tổng hợp) chỉ có trên Colab sau khi bạn `git push` lên GitHub.** Nếu chưa push, các lệnh mới sẽ báo `unrecognized arguments` hoặc `No module named`.

In [ ]:
%cd /content
!git clone https://github.com/ColorOfDreams/dds-edge.git 2>/dev/null || echo 'already cloned'
%cd /content/dds-edge
!git pull -q
!pip install -q timm onnx onnxruntime tabulate
!pip install -q -e . --no-deps
!git log --oneline -1

### Cell 4: giải nén dữ liệu và nối vào đúng chỗ
Tìm `sf_320x240*.zip` (hoặc `.rar`) trong My Drive, giải nén vào ổ đĩa của Colab (đọc ảnh từ ổ này nhanh hơn nhiều so với đọc trực tiếp từ Drive), rồi tạo liên kết để đường dẫn trong `manifest.csv` mở được ảnh.

**Kết quả cần thấy:** `20007 images; {'test': 4, 'train': 18, 'val': 4} drivers per split`. Nếu số khác đi thì dữ liệu chưa đúng. Nếu báo không thấy file, kiểm tra file đã upload lên *My Drive* chưa.

In [ ]:
# data: extract to the local disk (much faster than reading from Drive), then link to data/processed/sf_320x240
# accepts sf_320x240*.zip (preferred) or sf_320x240*.rar in My Drive
import os, glob, shutil, zipfile, subprocess
os.chdir('/content/dds-edge')
if not glob.glob('/content/sf_data/**/manifest.csv', recursive=True):
    arch = None
    for ext in ('zip', 'rar'):
        found = (glob.glob(f'/content/drive/MyDrive/sf_320x240*.{ext}')
                 or glob.glob(f'/content/drive/MyDrive/**/sf_320x240*.{ext}', recursive=True))
        if found:
            arch = found[0]
            break
    assert arch, 'Khong thay sf_320x240*.zip/.rar trong My Drive: da upload len Drive chua?'
    print('archive:', arch)
    shutil.rmtree('/content/sf_data', ignore_errors=True)
    os.makedirs('/content/sf_data')
    if arch.endswith('.zip'):
        with zipfile.ZipFile(arch) as z:
            for info in z.infolist():
                name = info.filename.replace(chr(92), '/')  # zips made on Windows use backslashes
                if name.endswith('/'):
                    continue
                dst = os.path.join('/content/sf_data', name)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(info) as f, open(dst, 'wb') as g:
                    shutil.copyfileobj(f, g)
    else:  # .rar needs the unrar tool
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'unrar'], check=True)
        subprocess.run(['unrar', 'x', '-idq', '-y', arch, '/content/sf_data/'], check=True)
hits = glob.glob('/content/sf_data/**/manifest.csv', recursive=True)
assert hits, 'Khong thay manifest.csv sau khi giai nen: ' + str(os.listdir('/content/sf_data')[:5])
src = os.path.dirname(hits[0])
os.makedirs('data/processed', exist_ok=True)
link = 'data/processed/sf_320x240'
if os.path.islink(link):  # stale link from an earlier run
    os.unlink(link)
assert not os.path.exists(link), f'{link} is a real folder, remove it first'
os.symlink(src, link)
import pandas as pd
m = pd.read_csv(link + '/manifest.csv')
print(len(m), 'images;', m.groupby('split')['group'].nunique().to_dict(), 'drivers per split')  # expect 20007; test 4, train 18, val 4

### Cell 5: lưu kết quả vào Drive
Biến thư mục `runs/` thành liên kết tới `MyDrive/dds-edge-runs/`. Mỗi lần train xong, các file (`best.pt`, `metrics.json`, `predictions.csv`, `confusion_matrix.csv`…) nằm sẵn trong Drive. Nếu Colab ngắt, bạn không mất `best.pt` (cần cho lượng tử hoá).

In [ ]:
# write runs/ straight to Drive so results survive a disconnect
%cd /content/dds-edge
!mkdir -p /content/drive/MyDrive/dds-edge-runs
!rm -rf runs
!ln -s /content/drive/MyDrive/dds-edge-runs runs
!ls -la runs | head -3

## B. Chạy thử

### Cell 6: train thử 1 epoch
Chạy MobileNetV3-Small đúng 1 epoch (tên run `smoke_colab`) chỉ để kiểm tra toàn bộ đường ống chạy thông: nạp dữ liệu, train, val, test, ghi kết quả. **Không dùng kết quả này để báo cáo.** Cờ `-u` làm dòng `ep 01 ...` hiện ngay khi xong epoch, không bị giữ lại trong bộ đệm.

**Kết quả cần thấy:** một dòng `ep 01 loss ... val_acc ... val_f1 ...`, rồi khối JSON có `test_macro_f1`. Khoảng 0,6 đến 0,7 là bình thường sau 1 epoch.

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --epochs 1 --manifest processed/sf_320x240/manifest.csv --workers 2 --name smoke_colab

## C. Train các cấu hình chính

### Cell 7: chọn mô hình và tên run (chạy cell này trước mọi cell train bên dưới)
Đặt hai biến dùng cho các lệnh sau:
- `ARCH`: kiến trúc: `mobilenet_v3_small`, `efficientnet_lite0` hoặc `resnet18`.
- `TAG`: tiền tố tên run, để mỗi thí nghiệm lưu vào thư mục riêng (ví dụ `mnv3`, `effl0`, `resnet18`).

Trong các lệnh `!python ...`, `{ARCH}` và `{TAG}` được notebook thay bằng giá trị của hai biến này. **Đừng viết `${TAG}` hay `$ARCH` kiểu bash**, vì sẽ làm tên run bị trống hoặc sai.

Điểm gốc để so sánh (test macro-F1, 10 epoch, dữ liệu sạch, đo trên Kaggle): MobileNetV3-Small 0,7524; EfficientNet-Lite0 0,7860; ResNet18 0,7517. Chạy lại trên Colab (GPU khác) có thể lệch nhẹ do GPU có nhiễu.

In [ ]:
ARCH = "mobilenet_v3_small"   # sau đó thử "efficientnet_lite0" / "resnet18"
TAG = "mnv3"                  # tiền tố tên run: "effl0" / "resnet18"
print(ARCH, TAG)

### Cell 8: baseline và số epoch lớn hơn
- **Baseline** (10 epoch, tham số mặc định) chỉ cần chạy lại nếu bạn chưa có `best.pt` của mô hình đó trong Drive. Kết quả cũ có ở runlog 3 đến 5.
- **20 epoch** (`..._ep20`): kiểm tra xem train lâu hơn có tốt hơn không. Nếu val và test macro-F1 tăng rõ (vượt khoảng 0,02 đến 0,03) thì dùng 20 epoch cho các thí nghiệm sau; nếu không thì giữ 10.

Chạy từng dòng một: bỏ `#` ở đúng dòng cần chạy.

In [ ]:
%cd /content/dds-edge
# baseline 10 epoch (chỉ chạy nếu chưa có best.pt của mô hình này)
# !python -u -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --name {TAG}_clean
# 20 epoch
# !python -u -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --epochs 20 --name {TAG}_ep20

## D. Phân tích lỗi (làm trước khi cải thiện)

Mỗi lần train xong, `train.py` lưu `runs/<tên run>/predictions.csv`: với **mỗi ảnh test** có đường dẫn, tài xế, nhãn thật, nhãn dự đoán và xác suất từng lớp. Nhờ file này biết mô hình sai ở đâu, thay vì đoán. Chỉ các run **train sau khi code mới được push** mới có file này.

### Cell 9: confusion matrix, F1 từng lớp, điểm từng tài xế
Đặt `RUN` là tên run cần xem. Cell in ra:
1. **Confusion matrix** (hàng = nhãn thật, cột = nhãn dự đoán), dạng số ảnh và dạng phần trăm theo hàng.
2. Precision, recall, F1 của từng lớp.
3. **Các nhầm lẫn thường gặp nhất** (ví dụ `talk_passenger → normal`).
4. **Độ chính xác từng tài xế** trong tập test: nếu một tài xế thấp hơn hẳn thì điểm thấp có thể do người đó khó, không hẳn do mô hình.

**Cần để ý:** `talk_passenger` và `normal` có bị nhầm với nhau không (nếu đúng thì chỉnh tham số khó giúp cả hai cùng lúc).

In [ ]:
RUN = "mnv3_clean"   # tên run cần phân tích (thư mục trong runs/)
%cd /content/dds-edge
!python -m dds_edge.dl.analyze --run {RUN}

### Cell 10: xem trực tiếp ảnh bị nhầm
Lưu một ảnh ghép các ảnh thuộc lớp `TRUE_CLS` mà mô hình đoán thành `PRED_CLS` (các ảnh nhầm với độ tự tin cao nhất xếp trước). Mỗi ảnh ghi mã tài xế và xác suất. File lưu ở `runs/<RUN>/mis_<TRUE>_as_<PRED>.jpg` và được hiện ngay bên dưới.

**Câu hỏi nên trả lời bằng mắt:** các ảnh `talk_passenger` bị đoán là `normal` có thật sự khác `normal` không (hướng đầu, tay, ánh mắt)? Nếu nhìn gần như giống hệt thì đó là giới hạn của dữ liệu chứ không phải lỗi mô hình.

In [ ]:
TRUE_CLS, PRED_CLS = "talk_passenger", "normal"
!python -m dds_edge.dl.analyze --run {RUN} --montage {TRUE_CLS} {PRED_CLS} --n 12 > /dev/null
from IPython.display import Image, display
display(Image(f"/content/dds-edge/runs/{RUN}/mis_{TRUE_CLS}_as_{PRED_CLS}.jpg"))

## E. Cải thiện: mất cân bằng lớp, augmentation và độ phân giải

Mỗi dòng đổi **một** thứ so với baseline cùng mô hình, để biết đúng thứ nào có tác dụng. So sánh với baseline qua **test macro-F1** và đặc biệt **F1 của `talk_passenger` và `normal`**, đồng thời kiểm tra các lớp khác (nhất là `phone`) không bị tụt.

### Cell 11: các thí nghiệm cải thiện
| Dòng | Ý nghĩa |
|---|---|
| `cw_sqrt` | **Class weight** căn bậc hai: loss nặng hơn cho lớp ít ảnh, nhẹ nhàng |
| `cw_inv` | Class weight nghịch đảo hoàn toàn: ép mạnh hơn (lớp `phone` bị giảm trọng số nhiều, có thể làm F1 của nó tụt) |
| `smp_sqrt` | **Sampler cân bằng** (căn bậc hai): mỗi epoch lấy lớp nhỏ nhiều lần hơn, mỗi lần với một phiên bản augmentation khác (oversampling có augmentation, thay cho SMOTE) |
| `smp_bal` | Sampler cân bằng hoàn toàn: mọi lớp được lấy với tần suất bằng nhau |
| `aug2` | **Augmentation mạnh hơn**: cắt rộng hơn, xoay 12°, đổi màu mạnh hơn, làm mờ, xoá ngẫu nhiên một mảng ảnh. Mục đích: giảm việc mô hình học thuộc đặc điểm từng tài xế (val cao hơn test nhiều) |
| `aug2_cw` | Kết hợp augmentation mạnh và class weight sqrt |
| `sz160` | Ảnh vào **nhỏ hơn** (160×160): rẻ nhất, nhanh nhất lúc suy luận, để xem mất bao nhiêu độ chính xác |
| `sz288x216` | Ảnh vào **chữ nhật 288×216**, giữ tỉ lệ 4:3 (không bị méo như ảnh vuông 224). Tốn khoảng 1,26 lần tính toán so với 224×224 |
| `sz320x240` | **Độ phân giải gốc 320×240**, không méo, không phóng to. Tốn khoảng 1,55 lần tính toán so với 224×224 (MobileNetV3-S: 55 triệu lên 86 triệu MACs). Lớn hơn 320×240 chỉ phóng to ảnh, không thêm chi tiết |

**Cách nhập `--img_size`:** một số (`224` = ảnh vuông 224×224) hoặc `RỘNGxCAO` (`320x240` = rộng 320, cao 240). Các dòng `sz...` trả lời câu hỏi: **tăng tốc độ tính toán (ảnh lớn hơn) thì độ chính xác tăng bao nhiêu?** Sau đó lượng tử hoá INT8 (phần G) để xem INT8 bù lại tốc độ được bao nhiêu.

Không dùng cùng lúc `class_weight` và `sampler` (sẽ bù mất cân bằng hai lần), trừ khi muốn thử riêng.

In [ ]:
%cd /content/dds-edge
# mất cân bằng lớp
# !python -u -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --class_weight sqrt --name {TAG}_cw_sqrt
# !python -u -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --class_weight inverse --name {TAG}_cw_inv
# !python -u -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --sampler sqrt --name {TAG}_smp_sqrt
# !python -u -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --sampler balanced --name {TAG}_smp_bal
# augmentation mạnh hơn
# !python -u -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --aug_level 2 --name {TAG}_aug2
# !python -u -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --aug_level 2 --class_weight sqrt --name {TAG}_aug2_cw
# độ phân giải ảnh vào (đổi tốc độ lấy độ chính xác)
# !python -u -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --img_size 160 --name {TAG}_sz160
# !python -u -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --img_size 288x216 --name {TAG}_sz288x216
# !python -u -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --img_size 320x240 --name {TAG}_sz320x240

## F. Kiểm tra `best.pt` trước khi lượng tử hoá

### Cell 12: danh sách run và file `best.pt`
Lượng tử hoá cần file `best.pt` (trọng số tốt nhất) của từng run. Cell này liệt kê **mọi run trong `runs/`**, cho biết run nào đã có `best.pt`, kích thước, và điểm test, để bạn biết run nào đã sẵn sàng.

- Cột `best.pt` = `OK`: có thể lượng tử hoá.
- Cột `best.pt` = `THIẾU`: run này chưa có checkpoint (ví dụ train trên Kaggle mà chưa tải về). Phải copy `best.pt` vào `runs/<tên run>/` trong Drive, hoặc train lại.

**Chưa chạy lượng tử hoá cho các run `THIẾU`.**

In [ ]:
import json, pathlib
rows = []
for d in sorted(pathlib.Path('/content/dds-edge/runs').iterdir()):
    if not d.is_dir():
        continue
    ck = d / 'best.pt'
    m = json.loads((d / 'metrics.json').read_text()) if (d / 'metrics.json').exists() else {}
    rows.append((d.name, 'OK' if ck.exists() else 'THIẾU',
                 f"{ck.stat().st_size / 1e6:.1f} MB" if ck.exists() else '-',
                 f"{m['test_macro_f1']:.4f}" if m else '-'))
print(f"{'run':28s}{'best.pt':10s}{'kích thước':14s}test macro-F1")
for r in rows:
    print(f"{r[0]:28s}{r[1]:10s}{r[2]:14s}{r[3]}")

## G. Lượng tử hoá INT8 và đo tốc độ

**Lượng tử hoá** là đổi trọng số (và/hoặc giá trị trung gian) của mô hình từ số thực 32 bit (FP32) sang số nguyên 8 bit (INT8) để mô hình **nhỏ hơn khoảng 4 lần và thường chạy nhanh hơn** trên chip edge. Đây là lượng tử hoá **sau huấn luyện** (post-training): không train lại.

**Các kiểu lượng tử hoá có trong code** (cờ của `export_quantize`):

| Cờ | Lựa chọn | Ý nghĩa |
|---|---|---|
| `--quant` | `static` (mặc định) | Cả trọng số **và** giá trị trung gian đều thành INT8. Cần ảnh mẫu (`--calib`) để đo khoảng giá trị. Thường chính xác nhất và nhanh nhất trên chip |
| | `dynamic` | Chỉ trọng số là INT8, giá trị trung gian lượng tử hoá ngay lúc chạy, **không cần ảnh mẫu**. Với mạng tích chập trên CPU thường **chậm hơn** (nhiều khi chậm hơn cả FP32) |
| `--calib_method` | `minmax` (mặc định) | Khoảng giá trị = nhỏ nhất đến lớn nhất. Nhạy với giá trị ngoại lai |
| | `entropy` | Chọn khoảng làm ít mất thông tin nhất (KL divergence), thường tốt hơn khi có ngoại lai |
| | `percentile` | Cắt bớt phần đuôi (ví dụ 99,999%), bỏ ngoại lai |
| `--quant_format` | `qdq` (mặc định) | Chèn các nút Quantize/Dequantize quanh toán tử; tương thích với nhiều chip và công cụ |
| | `qoperator` | Dùng toán tử đã lượng tử hoá sẵn; gọn hơn, nhưng tuỳ thư viện hỗ trợ |
| `--per_channel` | bật (mặc định) | Mỗi kênh đầu ra của lớp tích chập có một hệ số riêng: **chính xác hơn**, nhưng một số chip không hỗ trợ. `--no-per_channel` là một hệ số cho cả lớp |
| `--calib` | 200 (mặc định) | Số ảnh train dùng để đo khoảng giá trị. Nhiều hơn thì ổn định hơn (ví dụ 500) |
| `--threads` | 1 | Số luồng CPU khi đo. Dùng 1 luồng để so sánh công bằng giữa các mô hình |
| `--tag` | tuỳ chọn | Hậu tố tên file để lưu nhiều biến thể của cùng một run (`model_int8_<tag>.onnx`) |

**Kết quả đo cho mỗi bản (FP32 và INT8):** macro-F1 và accuracy, kích thước file (MB), **độ trễ một ảnh** (trung vị và p95, mili giây, batch 1), **FPS** (= 1000 / độ trễ trung vị), thông lượng khi chạy 16 ảnh một lúc (ảnh/giây). Cùng với số tham số và số phép tính (MACs).

**Cảnh báo quan trọng khi đọc số:**
- Đây là tốc độ trên **CPU của máy Colab**, không phải trên chip edge. Chỉ dùng để **xếp hạng tương đối** giữa các mô hình; FPS tuyệt đối trên chip thật phải đo trên chính thiết bị đó.
- Muốn so sánh công bằng thì **đo tất cả mô hình trong cùng một phiên Colab** (CPU của Colab có thể khác nhau giữa các phiên).
- INT8 chỉ nhanh hơn rõ trên CPU có tập lệnh hỗ trợ số nguyên 8 bit (VNNI/AVX512). Trên CPU không có, INT8 có thể nhỏ hơn nhưng không nhanh hơn.

### Cell 13: lượng tử hoá một run bằng cấu hình mặc định (static, minmax, qdq, per-channel)
Đặt `QRUN` là run đã có `best.pt` (xem cell 12). Cell xuất ONNX FP32, lượng tử hoá, rồi đo cả hai bản trên **toàn bộ ảnh test**. Kết quả ghi vào `runs/<QRUN>/quant_results.json`. Nếu run chưa có `best.pt`, cell báo rõ và dừng.

**Cần để ý ở kết quả:**
- `fp32.macro_f1` và `int8.macro_f1`: INT8 tụt bao nhiêu. Giả định ban đầu: tụt quá khoảng 2 điểm thì mô hình **không qua** cổng khả thi.
- `size_mb`: INT8 phải nhỏ hơn FP32 khoảng 3 đến 4 lần.
- `latency_ms_median`, `fps_median`: INT8 có nhanh hơn không (xem cảnh báo bên trên).

In [ ]:
QRUN = "mnv3_clean"   # run cần lượng tử hoá (phải có best.pt, xem cell 12)
%cd /content/dds-edge
!python -u -m dds_edge.dl.export_quantize --run {QRUN} --manifest processed/sf_320x240/manifest.csv --threads 1

### Cell 14: thử các kiểu lượng tử hoá khác của cùng một run
So sánh các cách lượng tử hoá trên cùng một mô hình để chọn cách tốt nhất (ít tụt F1 nhất hoặc nhanh nhất). Mỗi dòng ghi ra file riêng nhờ `--tag`, nên không ghi đè lên kết quả của cell 13. Bỏ `#` từng dòng.

Gợi ý theo thứ tự: `entropy` và `percentile` (cách chọn khoảng giá trị), `no_pc` (tắt per-channel, thường tụt F1 nhiều hơn), `calib500` (nhiều ảnh mẫu hơn), `qop` (định dạng khác), `dyn` (lượng tử hoá động, để thấy nó chậm hơn).

In [ ]:
%cd /content/dds-edge
# cách chọn khoảng giá trị của activation
# !python -u -m dds_edge.dl.export_quantize --run {QRUN} --manifest processed/sf_320x240/manifest.csv --threads 1 --calib_method entropy --tag entropy
# !python -u -m dds_edge.dl.export_quantize --run {QRUN} --manifest processed/sf_320x240/manifest.csv --threads 1 --calib_method percentile --tag percentile
# per-channel tắt, nhiều ảnh mẫu hơn, định dạng khác, lượng tử hoá động
# !python -u -m dds_edge.dl.export_quantize --run {QRUN} --manifest processed/sf_320x240/manifest.csv --threads 1 --no-per_channel --tag no_pc
# !python -u -m dds_edge.dl.export_quantize --run {QRUN} --manifest processed/sf_320x240/manifest.csv --threads 1 --calib 500 --tag calib500
# !python -u -m dds_edge.dl.export_quantize --run {QRUN} --manifest processed/sf_320x240/manifest.csv --threads 1 --quant_format qoperator --tag qop
# !python -u -m dds_edge.dl.export_quantize --run {QRUN} --manifest processed/sf_320x240/manifest.csv --threads 1 --quant dynamic --tag dyn

## H. Tổng hợp

### Cell 15: bảng và biểu đồ so sánh mọi run
Đọc tất cả `runs/*/metrics.json` và `quant_results*.json`, tạo:
- `reports/summary.csv` và `reports/summary.md`: mỗi run một dòng (và thêm một dòng cho mỗi bản INT8) với số tham số, MACs, kích thước MB, độ trễ, FPS, val và test macro-F1, F1 sau INT8, mức tụt do INT8, F1 của `normal` và `talk_passenger`.
- `reports/summary_plot.png`: **test macro-F1 theo độ trễ** (mỗi điểm một mô hình, màu theo FP32 hoặc INT8). Mô hình nằm càng **trên và bên trái** càng tốt (chính xác hơn và nhanh hơn). Nếu chưa có số đo độ trễ, biểu đồ dùng số phép tính (MACs) thay thế.

Cell cũng copy các file này sang Drive (`MyDrive/dds-edge-reports/`) để không mất.

In [ ]:
%cd /content/dds-edge
!python -m dds_edge.dl.summarize --runs_dir runs --out_dir reports
from IPython.display import Image, display
import os
if os.path.exists('reports/summary_plot.png'):
    display(Image('reports/summary_plot.png'))
!mkdir -p /content/drive/MyDrive/dds-edge-reports && cp -r reports/. /content/drive/MyDrive/dds-edge-reports/